In [ ]:
import os

# 设置 Hugging Face 镜像源a
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"


In [2]:
import cv2
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler,StableDiffusionPipeline
from diffusers.utils import load_image,make_image_grid
import numpy as np
import torch
from transformers import AutoImageProcessor, UperNetForSemanticSegmentation
from peft import LoraConfig, get_peft_model
import random
import ptp_utils_max_merge as ptp_utils

In [3]:
device = torch.device("cuda:1")
prefix="test53"

In [4]:
#Available modes Pose, Depth, HED, Canny, Seg

# pose,depth,canny,hed,seg,normal
controlnet_libs = {
    'pose': "lllyasviel/sd-controlnet-openpose",
    'scribble': "lllyasviel/sd-controlnet-scribble",
    'canny': "lllyasviel/sd-controlnet-canny",
    'hed': "lllyasviel/sd-controlnet-hed",
    'depth': "lllyasviel/sd-controlnet-depth",
    'seg' : "lllyasviel/sd-controlnet-seg",
    'normal': "lllyasviel/sd-controlnet-normal",
    'mlsd': "lllyasviel/sd-controlnet-mlsd",
}

def load_pipeline(mode_1, mode_2,device=device):

    controlnet_1 = ControlNetModel.from_pretrained(controlnet_libs[mode_1],torch_dtype=torch.float32,local_files_only=False)
    controlnet_2 = ControlNetModel.from_pretrained(controlnet_libs[mode_2],torch_dtype=torch.float32,local_files_only=False)
    
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet_2,torch_dtype=torch.float32)
    
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    pipe.controlnet1=controlnet_1.to(device)
    pipe.controlnet2= controlnet_2.to(device)

    pipe=pipe.to(device)

    return pipe



## complementary任务——hed_depth

In [5]:
# # mode要与读入的图片相对应
# mode_1 = 'hed'
# mode_2 = 'depth'
# pipe= load_pipeline(mode_1, mode_2)

# # 设定好种子，确保可重复实验
# seed=0
# g_cpu = torch.Generator().manual_seed(seed)
# latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [6]:
# control1_folder = "./data_picked_coco/hed_images/"
# control2_folder = "./data_picked_coco/depth_images/"
# prompt_folder = "./data_picked_coco/txts/"
# control1_files = [f for f in os.listdir(control1_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# control2_files = [f for f in os.listdir(control2_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# prompt_files = [f for f in os.listdir(prompt_folder) if f.endswith(('.txt'))]


# def extract_number(filename):
#     if '_' in filename:
#         return int(filename.split('_')[0])
#     return int(os.path.splitext(filename)[0])

# control1_files.sort(key=extract_number)
# control2_files.sort(key=extract_number)
# prompt_files.sort(key=extract_number)

# min_length = min(len(control1_files), len(control2_files), len(prompt_files))


# for i in range(500):
#     control1_path = os.path.join(control1_folder, control1_files[i])
#     control2_path = os.path.join(control2_folder, control2_files[i])
#     prompt_path = os.path.join(prompt_folder, prompt_files[i])

#     prompts=[]
#     control1=load_image(control1_path)
#     control2=load_image(control2_path)
#     with open(prompt_path, 'r', encoding='utf-8') as f:
#         prompts.append(next((line.strip() for line in f if line.strip()), ''))


#     negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

#     with torch.amp.autocast("cuda"):
#         image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
#                                                         prompts, 
#                                                         negative_prompts, 
#                                                         latent=latent, 
#                                                         num_inference_steps=50, 
#                                                         guidance_scale=7.5, 
#                                                         generator=g_cpu, 
#                                                         control1=control1,
#                                                         control2=control2,
#                                                         low_resource=False,
#                                                         thres=1/256,)
    
#     a=i//100
#     results_dir="./results/complementary/"+str(a)+"/"+prefix+"_hed_depth"
#     if not os.path.exists(results_dir):
#         os.makedirs(results_dir)


#     # 提取文件名（去除扩展名）
#     name1 = os.path.splitext(control1_files[i])[0]
#     name2 = os.path.splitext(control2_files[i])[0]
#     separator = "_and_"

#     # 生成新的文件名
#     new_filename = f"{name1}{separator}{name2}.png"

#     output_path = os.path.join(results_dir,new_filename)
#     ptp_utils.save_images(images=image,results_path=output_path)





## complementary任务——seg_depth

In [7]:
# # mode要与读入的图片相对应
# mode_1 = 'seg'
# mode_2 = 'depth'
# pipe= load_pipeline(mode_1, mode_2)

# # 设定好种子，确保可重复实验
# seed=0
# g_cpu = torch.Generator().manual_seed(seed)
# latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [8]:
# control1_folder = "./data_picked_coco/seg_images/"
# control2_folder = "./data_picked_coco/depth_images/"
# prompt_folder = "./data_picked_coco/txts/"
# control1_files = [f for f in os.listdir(control1_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# control2_files = [f for f in os.listdir(control2_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# prompt_files = [f for f in os.listdir(prompt_folder) if f.endswith(('.txt'))]


# def extract_number(filename):
#     if '_' in filename:
#         return int(filename.split('_')[0])
#     return int(os.path.splitext(filename)[0])

# control1_files.sort(key=extract_number)
# control2_files.sort(key=extract_number)
# prompt_files.sort(key=extract_number)

# min_length = min(len(control1_files), len(control2_files), len(prompt_files))


# for i in range(500):
#     control1_path = os.path.join(control1_folder, control1_files[i])
#     control2_path = os.path.join(control2_folder, control2_files[i])
#     prompt_path = os.path.join(prompt_folder, prompt_files[i])


#     prompts=[]
#     control1=load_image(control1_path)
#     control2=load_image(control2_path)
#     with open(prompt_path, 'r', encoding='utf-8') as f:
#         prompts.append(next((line.strip() for line in f if line.strip()), ''))


#     negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

#     with torch.amp.autocast("cuda"):
#         image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
#                                                         prompts, 
#                                                         negative_prompts, 
#                                                         latent=latent, 
#                                                         num_inference_steps=50, 
#                                                         guidance_scale=7.5, 
#                                                         generator=g_cpu, 
#                                                         control1=control1,
#                                                         control2=control2,
#                                                         low_resource=False,
#                                                         thres=1/256,)
    
#     a=i//100
#     results_dir="./results/complementary/"+str(a)+"/"+prefix+"_seg_depth"
#     if not os.path.exists(results_dir):
#         os.makedirs(results_dir)


#     # 提取文件名（去除扩展名）
#     name1 = os.path.splitext(control1_files[i])[0]
#     name2 = os.path.splitext(control2_files[i])[0]
#     separator = "_and_"

#     # 生成新的文件名
#     new_filename = f"{name1}{separator}{name2}.png"

#     output_path = os.path.join(results_dir,new_filename)
#     ptp_utils.save_images(images=image,results_path=output_path)





## complementary任务——hed_seg

In [9]:
# # mode要与读入的图片相对应
# mode_1 = 'hed'
# mode_2 = 'seg'
# pipe= load_pipeline(mode_1, mode_2)

# # 设定好种子，确保可重复实验
# seed=0
# g_cpu = torch.Generator().manual_seed(seed)
# latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [10]:
# control1_folder = "./data_picked_coco/hed_images/"
# control2_folder = "./data_picked_coco/seg_images/"
# prompt_folder = "./data_picked_coco/txts/"
# control1_files = [f for f in os.listdir(control1_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# control2_files = [f for f in os.listdir(control2_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
# prompt_files = [f for f in os.listdir(prompt_folder) if f.endswith(('.txt'))]


# def extract_number(filename):
#     if '_' in filename:
#         return int(filename.split('_')[0])
#     return int(os.path.splitext(filename)[0])

# control1_files.sort(key=extract_number)
# control2_files.sort(key=extract_number)
# prompt_files.sort(key=extract_number)

# min_length = min(len(control1_files), len(control2_files), len(prompt_files))


# for i in range(500):
#     control1_path = os.path.join(control1_folder, control1_files[i])
#     control2_path = os.path.join(control2_folder, control2_files[i])
#     prompt_path = os.path.join(prompt_folder, prompt_files[i])


#     prompts=[]
#     control1=load_image(control1_path)
#     control2=load_image(control2_path)
#     with open(prompt_path, 'r', encoding='utf-8') as f:
#         prompts.append(next((line.strip() for line in f if line.strip()), ''))


#     negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

#     with torch.amp.autocast("cuda"):
#         image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
#                                                         prompts, 
#                                                         negative_prompts, 
#                                                         latent=latent, 
#                                                         num_inference_steps=50, 
#                                                         guidance_scale=7.5, 
#                                                         generator=g_cpu, 
#                                                         control1=control1,
#                                                         control2=control2,
#                                                         low_resource=False,
#                                                         thres=1/256,)
    
#     a=i//100
#     results_dir="./results/complementary/"+str(a)+"/"+prefix+"_hed_seg"
#     if not os.path.exists(results_dir):
#         os.makedirs(results_dir)


#     # 提取文件名（去除扩展名）
#     name1 = os.path.splitext(control1_files[i])[0]
#     name2 = os.path.splitext(control2_files[i])[0]
#     separator = "_and_"

#     # 生成新的文件名
#     new_filename = f"{name1}{separator}{name2}.png"

#     output_path = os.path.join(results_dir,new_filename)
#     ptp_utils.save_images(images=image,results_path=output_path)





## contradictory任务——pose_depth

In [ ]:
# mode要与读入的图片相对应
mode_1 = 'pose'
mode_2 = 'depth'
pipe= load_pipeline(mode_1, mode_2)

# 设定好种子，确保可重复实验
seed=0
g_cpu = torch.Generator().manual_seed(seed)
latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [ ]:
prompt = "a man in a garden , best quality"
prompts=[prompt]
negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

control1_folder = "./data_openpose_selected/"
control2_folder = "./data_background/depth_background/"
control1_files = [f for f in os.listdir(control1_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
control2_files = [f for f in os.listdir(control2_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

control1_files = control1_files[0:10]
control2_files = control2_files[0:50]


for i,random_control2_file in enumerate(control2_files):
    for random_control1_file in control1_files:

        control1_path = os.path.join(control1_folder, random_control1_file)
        control2_path = os.path.join(control2_folder, random_control2_file)


        control1=load_image(control1_path)
        control2=load_image(control2_path)

        with torch.amp.autocast("cuda"):
            image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
                                                            prompts, 
                                                            negative_prompts, 
                                                            latent=latent, 
                                                            num_inference_steps=50, 
                                                            guidance_scale=7.5, 
                                                            generator=g_cpu, 
                                                            control1=control1,
                                                            control2=control2,
                                                            low_resource=False,
                                                            thres=1/256,)
        
        a=i//10
        results_dir="./results/contradictory/"+str(a)+"/"+prefix+"_pose_depth"
        if not os.path.exists(results_dir):
            os.makedirs(results_dir)


        # 提取文件名（去除扩展名）
        name1 = os.path.splitext(random_control1_file)[0]
        name2 = os.path.splitext(random_control2_file)[0]
        separator = "_and_"

        # 生成新的文件名
        new_filename = f"{name1}{separator}{name2}.png"

        output_path = os.path.join(results_dir,new_filename)
        ptp_utils.save_images(images=image,results_path=output_path)





## contradictory任务——pose_seg

In [ ]:
# mode要与读入的图片相对应
mode_1 = 'pose'
mode_2 = 'seg'
pipe= load_pipeline(mode_1, mode_2)

# 设定好种子，确保可重复实验
seed=0
g_cpu = torch.Generator().manual_seed(seed)
latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [ ]:
prompt = "a man in a garden , best quality"
prompts=[prompt]
negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

control1_folder = "./data_openpose_selected/"
control2_folder = "./data_background/seg_background/"
control1_files = [f for f in os.listdir(control1_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
control2_files = [f for f in os.listdir(control2_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

control1_files = control1_files[0:10]
control2_files = control2_files[0:50]


for i,random_control2_file in enumerate(control2_files):
    for random_control1_file in control1_files:

        control1_path = os.path.join(control1_folder, random_control1_file)
        control2_path = os.path.join(control2_folder, random_control2_file)


        control1=load_image(control1_path)
        control2=load_image(control2_path)

        with torch.amp.autocast("cuda"):
            image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
                                                            prompts, 
                                                            negative_prompts, 
                                                            latent=latent, 
                                                            num_inference_steps=50, 
                                                            guidance_scale=7.5, 
                                                            generator=g_cpu, 
                                                            control1=control1,
                                                            control2=control2,
                                                            low_resource=False,
                                                            thres=1/256,)
            
        a=i//10
        results_dir="./results/contradictory/"+str(a)+"/"+prefix+"_pose_seg"
        if not os.path.exists(results_dir):
            os.makedirs(results_dir)


        # 提取文件名（去除扩展名）
        name1 = os.path.splitext(random_control1_file)[0]
        name2 = os.path.splitext(random_control2_file)[0]
        separator = "_and_"

        # 生成新的文件名
        new_filename = f"{name1}{separator}{name2}.png"

        output_path = os.path.join(results_dir,new_filename)
        ptp_utils.save_images(images=image,results_path=output_path)



